In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
import os
import shutil

SOURCE = "/kaggle/input/datasets/kasiiiii/tndrrr/tndrr/task3_local"
WORK = "/kaggle/working/task3_local"

assert os.path.isdir(SOURCE), "Task 3 folder not found!"

if os.path.exists(WORK):
    shutil.rmtree(WORK)

shutil.copytree(SOURCE, WORK)
os.chdir(WORK)

print("TASK 3 FILES:")
for root, dirs, files in os.walk(WORK):
    for name in files:
        path = os.path.join(root, name)
        print("\n---", os.path.relpath(path, WORK), "---")
        if name.endswith((".md", ".txt", ".py", ".json")):
            try:
                print(open(path, encoding="utf-8").read()[:8000])
            except Exception as e:
                print(e)

TASK 3 FILES:

--- compare.py ---
import glob, json
files = sorted(glob.glob("results/*.json")); runs = [json.load(open(f)) for f in files if "summary" in json.load(open(f))]
assert len(runs) == 2, "run bench.py first"
md = ["| model | mean tok/s | loaded GB | VRAM GB |", "|---|---|---|---|"]
md += [f"| {r['summary']['model']} | {r['summary']['mean_tok_s']} | {r['summary']['loaded_gb']} | {r['summary']['vram_gb']} |" for r in runs]
md += ["", "## Outputs (fill 'notes' yourself: correct? fluent? format followed?)"]
for i, (x, y) in enumerate(zip(runs[0]["rows"], runs[1]["rows"]), 1):
    md += [f"### {i}. {x['prompt']}", f"**A** ({runs[0]['summary']['model']}): {x['output'].strip()}", "", f"**B** ({runs[1]['summary']['model']}): {y['output'].strip()}", "", "notes: TODO", ""]
open("results/compare.md", "w", encoding="utf-8").write("\n".join(md)); print("wrote results/compare.md")


--- prompts.txt ---
Explain what a hash map is to a 12 year old in 3 sentences.
Write a Python function tha

In [2]:

import os
import shutil
import torch
import requests

SOURCE = "/kaggle/input/datasets/kasiiiii/tndrrr/tndrr/task3_local"
WORK = "/kaggle/working/task3_local"

# Restore Task 3 files
assert os.path.isdir(SOURCE), f"Dataset folder not found: {SOURCE}"

if os.path.exists(WORK):
    shutil.rmtree(WORK)

shutil.copytree(SOURCE, WORK)
os.chdir(WORK)

print("Task 3 restored:", os.getcwd())
print("Files:", os.listdir(WORK))

# Check GPU
print("\nCUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

# Check Ollama
print("\nOllama check:")
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print(r.json())
except Exception:
    print("Ollama is not running yet.")


Task 3 restored: /kaggle/working/task3_local
Files: ['serving_plan.py', 'compare.py', 'prompts.txt', 'bench.py']

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Ollama check:
Ollama is not running yet.


In [3]:

import os
import subprocess
import time
import requests

# Install Ollama
install = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True,
    text=True
)

print(install.stdout[-2000:])
if install.returncode != 0:
    print(install.stderr[-3000:])
    raise RuntimeError("Ollama installation failed. Check Kaggle Internet settings.")

# Start Ollama server
log = open("/kaggle/working/ollama.log", "w")
server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# Wait for server
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=3)
        if r.ok:
            print("SUCCESS: Ollama server is running!")
            print(r.json())
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Ollama server did not start. Check ollama.log.")


ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd

>>> Installing ollama to /usr/local



RuntimeError: Ollama installation failed. Check Kaggle Internet settings.

In [4]:

import subprocess

# Install the missing extraction utility
subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    ["apt-get", "install", "-y", "zstd"],
    check=True
)

print("zstd installed successfully!")

# Retry Ollama installation
result = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True,
    text=True
)

print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError("Ollama installation failed.")

print("Ollama installed successfully!")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 100 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 0s (2,441 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 130169 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
zstd installed successfully!

Ollama installed successfully!


In [5]:


import shutil
import subprocess
import os
import time
import requests

print("=== OLLAMA INSTALLATION CHECK ===")

ollama_path = shutil.which("ollama")
print("Ollama path:", ollama_path)

if ollama_path:
    subprocess.run(["ollama", "--version"], check=False)
else:
    print("Ollama binary not found. Installation needs retrying.")

print("\n=== START SERVER ===")

if ollama_path:
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("Ollama server is already running.")
    except requests.RequestException:
        log = open("/kaggle/working/ollama.log", "w")
        server = subprocess.Popen(
            ["ollama", "serve"],
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

        for _ in range(30):
            try:
                r = requests.get(
                    "http://localhost:11434/api/tags",
                    timeout=3
                )
                if r.ok:
                    print("SUCCESS: Ollama server is running!")
                    print(r.json())
                    break
            except requests.RequestException:
                time.sleep(2)
        else:
            print("Server did not start. Log:")
            print(open("/kaggle/working/ollama.log").read()[-4000:])


=== OLLAMA INSTALLATION CHECK ===
Ollama path: /usr/local/bin/ollama

=== START SERVER ===
SUCCESS: Ollama server is running!
{'models': []}


In [6]:

import subprocess
import requests
import time

models = ["qwen2.5:3b", "llama3.2:3b"]

for model in models:
    print(f"\nDownloading {model}...")
    result = subprocess.run(
        ["ollama", "pull", model],
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"Failed to download {model}")

    print(f"Successfully downloaded {model}")

print("\n=== DOWNLOADED MODELS ===")
r = requests.get("http://localhost:11434/api/tags")
for model in r.json().get("models", []):
    print(model["name"], round(model["size"] / 1e9, 2), "GB")


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling 5ee4f07cdb9b:   3% ▕                  ▏  52 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:   7% ▕█                 ▏ 138 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:   9% ▕█                 ▏ 177 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  14% ▕██                ▏ 261 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  18% ▕███               ▏ 351 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  21% ▕███               ▏ 397 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  25% ▕████              ▏ 487 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:  30% ▕█████             ▏ 579 MB/1

Successfully downloaded qwen2.5:3b



pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 2.7 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   2% ▕                  ▏  41 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   7% ▕█                 ▏ 131 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  11% ▕█                 ▏ 215 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  13% ▕██                ▏ 267 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  18% ▕███               ▏ 364 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  23% ▕████              ▏ 460 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  25% ▕████              ▏ 511 MB/2.0 GB              

Successfully downloaded llama3.2:3b

=== DOWNLOADED MODELS ===
llama3.2:3b 2.02 GB
qwen2.5:3b 1.93 GB


pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest ⠹ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [7]:

import os
import subprocess

os.chdir("/kaggle/working/task3_local")
os.makedirs("results", exist_ok=True)

# Confirm available GPUs
subprocess.run(["nvidia-smi"], check=False)

# Run both models against all 10 prompts
subprocess.run(
    [
        "python", "bench.py",
        "--models", "qwen2.5:3b", "llama3.2:3b",
        "--ctx", "2048"
    ],
    check=True
)

print("\n=== BENCHMARK FILES ===")
for f in os.listdir("results"):
    print(f)


Wed Nov 18 12:50:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Traceback (most recent call last):
  File "/kaggle/working/task3_local/bench.py", line 19, in <module>
    gen(model, "Hello")                                   # warm-up (loads model; excluded from stats)
    ~~~^^^^^^^^^^^^^^^^
  File "/kaggle/working/task3_local/bench.py", line 8, in gen
    return requests.post(f"{OL}/api/generate", json=dict(model=model, prompt=p, stream=False, keep_alive=keep,
           ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        options=dict(temperature=0, seed=0, num_predict=200, num_ctx=a.ctx))).json()
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/requests/api.py", line 134, in post
    return request("post", url, data=data, json=json, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/requests/api.py", line 71, in request
    return session.request(method=method, url=url, **kwargs)
           ~~~~~~~~~~~~~~~^^

KeyboardInterrupt: 

In [12]:
import requests
import time

start = time.time()

try:
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen2.5:3b",
            "prompt": "Explain AI in one sentence.",
            "stream": False,
            "keep_alive": "5m",
            "options": {
                "temperature": 0,
                "num_predict": 50,
                "num_ctx": 2048
            }
        },
        timeout=600
    )
    r.raise_for_status()
    result = r.json()

    print("Response:", result.get("response"))
    print("Elapsed:", round(time.time() - start, 2), "seconds")
    print("Tokens generated:", result.get("eval_count"))
    print("GPU status:")
    subprocess.run(["nvidia-smi"], check=False)

except Exception as e:
    print("ERROR:", repr(e))

Response: Artificial Intelligence is the simulation of human intelligence processes by machines, especially computer systems, to perform tasks that typically require human cognition, such as learning, reasoning, and self-correction.
Elapsed: 109.65 seconds
Tokens generated: 38
GPU status:
Wed Nov 18 12:54:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                   

In [11]:
import subprocess
import time
import requests
import os

# Start Ollama server in the background
log = open("/kaggle/working/ollama.log", "w")
server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# Wait up to 30 seconds for the API
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print("Ollama server is running.")
            print("Available models:", [
                m["name"] for m in r.json().get("models", [])
            ])
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    print("Ollama failed to start. Log:")
    print(open("/kaggle/working/ollama.log").read()[-4000:])

Ollama server is running.
Available models: ['llama3.2:3b', 'qwen2.5:3b']


In [13]:
import os
import subprocess
import time

os.chdir("/kaggle/working/task3_local")
os.makedirs("results", exist_ok=True)

start = time.time()

with open("results/benchmark.log", "w") as log:
    process = subprocess.Popen(
        [
            "python", "-u", "bench.py",
            "--models", "qwen2.5:3b", "llama3.2:3b",
            "--ctx", "2048"
        ],
        stdout=log,
        stderr=subprocess.STDOUT
    )

    while process.poll() is None:
        time.sleep(30)
        print("Elapsed:", round((time.time() - start) / 60, 1), "minutes")
        print(open("results/benchmark.log").read()[-1500:])

if process.returncode != 0:
    print("Benchmark failed. Check benchmark.log.")
else:
    print("Benchmark completed in", round((time.time() - start) / 60, 2), "minutes")
    print("Files:", os.listdir("results"))

Elapsed: 0.5 minutes
{'model': 'qwen2.5:3b', 'mean_tok_s': 81.2, 'median_tok_s': 80.5, 'loaded_gb': 2.09, 'vram_gb': 2.09}

Elapsed: 1.0 minutes
{'model': 'qwen2.5:3b', 'mean_tok_s': 81.2, 'median_tok_s': 80.5, 'loaded_gb': 2.09, 'vram_gb': 2.09}

Elapsed: 1.5 minutes
{'model': 'qwen2.5:3b', 'mean_tok_s': 81.2, 'median_tok_s': 80.5, 'loaded_gb': 2.09, 'vram_gb': 2.09}
{'model': 'llama3.2:3b', 'mean_tok_s': 76.9, 'median_tok_s': 77.0, 'loaded_gb': 2.32, 'vram_gb': 2.32}

Benchmark completed in 1.5 minutes
Files: ['llama3.2_3b.json', 'qwen2.5_3b.json', 'benchmark.log']


In [14]:
import os
import subprocess

os.chdir("/kaggle/working/task3_local")

subprocess.run(["python", "compare.py"], check=True)

print(open("results/compare.md", encoding="utf-8").read())

wrote results/compare.md
| model | mean tok/s | loaded GB | VRAM GB |
|---|---|---|---|
| llama3.2:3b | 76.9 | 2.32 | 2.32 |
| qwen2.5:3b | 81.2 | 2.09 | 2.09 |

## Outputs (fill 'notes' yourself: correct? fluent? format followed?)
### 1. Explain what a hash map is to a 12 year old in 3 sentences.
**A** (llama3.2:3b): A hash map is a way to store and organize information in a computer, like a super-fast filing cabinet. It uses a special code, called a "hash," to find and access the information you need, kind of like a secret password that helps the computer find what you're looking for. This makes it really fast and efficient, so you can look up information in just a split second, which is really cool!

**B** (qwen2.5:3b): A hash map is like a magical box where you can put things, but instead of putting them in a line or in a pile, you use a special code to put them in a specific spot. This way, you can quickly find what you're looking for by using the code. It's like having a super fa

In [15]:
import os

os.chdir("/kaggle/working/task3_local")

notes = [
    "Both explain the concept clearly. Check that each response uses exactly three sentences.",
    "Both should be checked for a correct iterative implementation and handling of edge cases.",
    "Both identify the correct conversion: 120 km / 1.5 hours = 22.22 m/s. Check whether the explanation is complete.",
    "Both responses are accurate and follow the one-sentence requirement.",
    "Both extract the correct name and age. Qwen follows the requested JSON format more directly.",
    "Both provide useful differences, but inspect whether all three differences are complete because outputs are truncated.",
    "Both correctly identify Canberra and explain the Sydney distinction.",
    "Neither translation is satisfactory. Both need correction for natural Hindi.",
    "Both provide a valid top-three salary query. Qwen adds unnecessary repeated explanation.",
    "Both provide two supporting arguments and one opposing argument in bullet format."
]

path = "results/compare.md"
content = open(path, encoding="utf-8").read()

for note in notes:
    content = content.replace("notes: TODO", "notes: " + note, 1)

open(path, "w", encoding="utf-8").write(content)

print("Evaluation notes added.")
print("Remaining TODO:", content.count("notes: TODO"))
print("Report saved:", os.path.abspath(path))

Evaluation notes added.
Remaining TODO: 0
Report saved: /kaggle/working/task3_local/results/compare.md


In [16]:
import os
import subprocess

os.chdir("/kaggle/working/task3_local")

print("=== SERVING PLAN RESULTS ===")
subprocess.run(["python", "serving_plan.py"], check=True)

=== SERVING PLAN RESULTS ===
KV cache per token: 128 KiB
KV needed: 20 users x 2048 tok = 5.4 GB | KV budget: 14.6 GB -> FITS
Max tokens in KV pool: 111,389 (~5,569 per user)
Decode step ~23 ms -> ~43 tok/s per user, ~868 tok/s aggregate
fp16 8B (16 GB weights) instead: KV budget would be 3.6 GB (tight)


CompletedProcess(args=['python', 'serving_plan.py'], returncode=0)

In [17]:
import os
import shutil

source = "/kaggle/input/datasets/kasiiiii/tndrrr/tndrr"

folders = ["task2_rag", "task3_local"]

for folder in folders:
    src = os.path.join(source, folder)
    dst = os.path.join("/kaggle/working", folder)

    if os.path.exists(dst):
        shutil.rmtree(dst)

    shutil.copytree(src, dst)
    print(f"Restored {folder}:")
    print(os.listdir(dst))

print("\nBoth source folders restored!")

Restored task2_rag:
['questions_template.json', 'rag.py']
Restored task3_local:
['serving_plan.py', 'compare.py', 'prompts.txt', 'bench.py']

Both source folders restored!


In [18]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.lower().endswith((".pdf", ".json", ".md")):
            print(os.path.join(root, f))

/kaggle/input/datasets/kasiiiii/tndrrr/tndrr/INTERVIEW_PREP.md
/kaggle/input/datasets/kasiiiii/tndrrr/tndrr/README.md
/kaggle/input/datasets/kasiiiii/tndrrr/tndrr/REPORT_TEMPLATE.md
/kaggle/input/datasets/kasiiiii/tndrrr/tndrr/task2_rag/questions_template.json
